# SpaceX Falcon 9 – Interactive Visual Analytics with Folium
Mark all launch sites, mark the success/failed launches for each site, and calculate distances between a launch site and its proximities.

In [1]:
import folium
import pandas as pd
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians
import os

In [2]:
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv"
spacex_df = pd.read_csv("data/spacex_launch_geo.csv" if os.path.exists("data/spacex_launch_geo.csv") else URL)
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


In [3]:
os.makedirs('maps', exist_ok=True)

## Task 1: Mark all launch sites on a map

In [4]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=4)

# NASA Johnson Space Center as a reference point
folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center')).add_to(site_map)
folium.map.Marker(nasa_coordinate, icon=DivIcon(icon_size=(160, 20), icon_anchor=(-6, 6),
    html='<div style="font-size: 12px; color:#d35400; white-space: nowrap;"><b>NASA JSC</b></div>')).add_to(site_map)

for _, row in launch_sites_df.iterrows():
    coord = [row['Lat'], row['Long']]
    folium.Circle(coord, radius=1000, color='#000000', fill=True).add_child(folium.Popup(row['Launch Site'])).add_to(site_map)
    folium.map.Marker(coord, icon=DivIcon(icon_size=(160, 20), icon_anchor=(-6, 6),
        html=f'<div style="font-size: 12px; color:#d35400; white-space: nowrap;"><b>{row["Launch Site"]}</b></div>')).add_to(site_map)

site_map.save('maps/task1_launch_sites.html')
site_map

All launch sites are close to the coast, and they are in the south of the US (close to the equator).

## Task 2: Mark the success/failed launches for each site

In [5]:
marker_cluster = MarkerCluster()
spacex_df['marker_color'] = spacex_df['class'].apply(lambda c: 'green' if c == 1 else 'red')

site_map2 = folium.Map(location=[28.5623, -80.5774], zoom_start=5)
site_map2.add_child(marker_cluster)
for _, record in spacex_df.iterrows():
    marker = folium.Marker(location=[record['Lat'], record['Long']],
                           icon=folium.Icon(color='white', icon_color=record['marker_color'], icon='rocket', prefix='fa'),
                           popup=f"{record['Launch Site']} – class {record['class']}")
    marker_cluster.add_child(marker)

site_map2.save('maps/task2_outcome_clusters.html')
site_map2

In [6]:
spacex_df.groupby('Launch Site')['class'].agg(['sum', 'count', 'mean'])

,sum,count,mean
Launch Site,,,
CCAFS LC-40,7,26,0.269231
CCAFS SLC-40,3,7,0.428571
KSC LC-39A,10,13,0.769231
VAFB SLC-4E,4,10,0.400000


## Task 3: Calculate the distances between a launch site to its proximities

In [7]:
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

In [8]:
launch_site = launch_sites_df[launch_sites_df['Launch Site'] == 'CCAFS SLC-40'].iloc[0]
site_lat, site_lon = launch_site['Lat'], launch_site['Long']

# Points of interest read with MousePosition on the map
proximities = {
    'Coastline': (28.56250, -80.56795),
    'KSC LC-39A launch site': (28.57325457, -80.64689529),
    'Cape Canaveral (city)': (28.39220, -80.60770),
    'Titusville (city)': (28.61220, -80.80760),
    'Melbourne (city)': (28.08360, -80.60810),
}

site_map3 = folium.Map(location=[site_lat, site_lon], zoom_start=10)
site_map3.add_child(MousePosition(position='topright', separator=' Long: ', prefix='Lat:',
                                  lat_formatter="function(num) {return L.Util.formatNum(num, 5);};",
                                  lng_formatter="function(num) {return L.Util.formatNum(num, 5);};"))
folium.Circle([site_lat, site_lon], radius=1000, color='#000000', fill=True).add_to(site_map3)
folium.map.Marker([site_lat, site_lon], icon=DivIcon(icon_size=(160, 20), icon_anchor=(-6, 6),
    html='<div style="font-size: 12px; color:#d35400; white-space: nowrap;"><b>CCAFS SLC-40</b></div>')).add_to(site_map3)

distances = {}
for name, (lat, lon) in proximities.items():
    d = calculate_distance(site_lat, site_lon, lat, lon)
    distances[name] = round(d, 2)
    folium.Marker([lat, lon], icon=DivIcon(icon_size=(160, 20), icon_anchor=(-6, 6),
        html=f'<div style="font-size: 12px; color:#d35400; white-space: nowrap;"><b>{name}: {d:.2f} KM</b></div>')).add_to(site_map3)
    folium.PolyLine(locations=[[site_lat, site_lon], [lat, lon]], weight=2).add_to(site_map3)

site_map3.save('maps/task3_proximities.html')
print(distances)
site_map3

{'Coastline': 0.87, 'KSC LC-39A launch site': 6.94, 'Cape Canaveral (city)': 19.26, 'Titusville (city)': 23.19, 'Melbourne (city)': 53.43}


Launch sites are very close to the coastline (under 1 km) so rockets fly over the ocean, and they keep a distance of tens of kilometres from cities.